In [0]:
%sql

INSERT OVERWRITE proyecto_final.silver.universidades (
    id_universidad,
    nombre_universidad,
    unidad_academica,
    sede,
    direccion,
    barrio,
    comuna,
    telefono,
    fax,
    sitio_web,
    longitud,
    latitud,
    coordenadas,
    tipo_entidad
)
WITH datos_limpios AS (
    SELECT 
        id AS id_universidad,
        LOWER(REPLACE(TRIM(nombre), '"', '')) AS nombre_universidad,
        LOWER(REPLACE(TRIM(unidad_aca), '"', '')) AS unidad_academica,
        LOWER(REPLACE(TRIM(sede), '"', '')) AS sede,
        LOWER(REPLACE(TRIM(direccion), '"', '')) AS direccion,
        LOWER(REPLACE(TRIM(barrio), '"', '')) AS barrio,
        try_cast(TRIM(REPLACE(REPLACE(LOWER(TRIM(comuna)), '"', ''), 'comuna', '')) AS INT) AS comuna,
        REPLACE(TRIM(telefono), '"', '') AS telefono,
        REPLACE(TRIM(fax), '"', '') AS fax,
        REPLACE(TRIM(web), '"', '') AS sitio_web,
        CAST(SPLIT(TRIM(REPLACE(REPLACE(REPLACE(UPPER(geometry), 'POINT', ''), '(', ''), ')', '')), ' ')[0] AS DOUBLE) AS longitud,
        CAST(SPLIT(TRIM(REPLACE(REPLACE(REPLACE(UPPER(geometry), 'POINT', ''), '(', ''), ')', '')), ' ')[1] AS DOUBLE) AS latitud,
        geometry AS coordenadas,
        'universidad' AS tipo_entidad
    FROM proyecto_final.raw.universidades_bronze
    WHERE nombre IS NOT NULL 
      AND geometry IS NOT NULL
),
datos_deduplicados AS (
    SELECT *,
        ROW_NUMBER() OVER (PARTITION BY nombre_universidad, coordenadas ORDER BY id_universidad) AS rn
    FROM datos_limpios
)
SELECT 
    id_universidad,
    nombre_universidad,
    unidad_academica,
    sede,
    direccion,
    barrio,
    comuna,
    telefono,
    fax,
    sitio_web,
    longitud,
    latitud,
    coordenadas,
    tipo_entidad
FROM datos_deduplicados
WHERE rn = 1;